In [1]:
import torch
from transformers import AutoTokenizer, AutoModel

# Load the ChemBERTa-77M-MLM model and tokenizer
model_name = "DeepChem/ChemBERTa-77M-MTR"       # --->>> multitask regression
#model_name = "DeepChem/ChemBERTa-77M-MLM"      # ---->>> Embedding size is 384 trained on 77M on PubChem ...Masked Language Modeling 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

2026-06-30 10:46:44.912861: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-30 10:46:45.056346: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
import os
import pickle
from rdkit import Chem
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('./final_total.csv')
df

,CHEMBLID,smiles,ido_ic50,tdo_ic50
0,CHEMBL1098875,O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O,23.0,60.0
1,CHEMBL1209728,Cc1c(Br)oc2c1C(=O)C(=O)c1c-2ccc2c1CCCC2(C)C,10000.0,10000.0
2,CHEMBL1276265,O=C1c2ccc(Cl)cc2-n2c1nc1ccccc1c2=O,183.0,224200.0
3,CHEMBL1346056,Oc1ccccc1-c1nc2c3ccccc3c3ccccc3c2[nH]1,11790.0,12250.0
4,CHEMBL139935,O=[N+]([O-])c1cc(F)c2cccnc2c1O,14610.0,30230.0
...,...,...,...,...
755,CHEMBL5221020,O[C@@H]1CCCC[C@H]1CNc1c(Br)ccc2[nH]ncc12,1230.0,11240.0
756,CHEMBL5221102,Brc1cc(NC[C@H]2CCCN2)c2cn[nH]c2c1,56450.0,7450.0
757,CHEMBL578036,CC1(C)CCCc2c1ccc1c2C(=O)C(=O)c2c(CO)coc2-1,10000.0,450.0
758,CHEMBL584991,Nc1nonc1/C(=N/O)Nc1ccc(F)c(Cl)c1,50.0,10000.0


In [4]:
smiles_list = []
for i in range(len(df['smiles'])):
    smiles_list.append(df['smiles'][i])

In [5]:
len(smiles_list)

760

In [6]:
# Tokenize the SMILES strings
inputs = tokenizer(smiles_list, padding=True, truncation=True, return_tensors="pt")

In [7]:
len(inputs[1])

79

In [8]:
# Get the molecular embeddings
with torch.no_grad():
    outputs = model(**inputs)

In [9]:
outputs.last_hidden_state.shape

torch.Size([760, 79, 384])

In [10]:
# Extract the embeddings (CLS token representation)
embeddings_MTR = outputs.last_hidden_state[:, 0, :]

# Print the shape of embeddings
print("Molecular embeddings shape:", embeddings_MTR.shape)  # (num_molecules, embedding_dim)

Molecular embeddings shape: torch.Size([760, 384])


In [25]:
# Extract the embeddings (CLS token representation)
embeddings_MLM = outputs.last_hidden_state[:, 0, :]

# Print the shape of embeddings
print("Molecular embeddings shape:", embeddings_MLM.shape)  # (num_molecules, embedding_dim)

Molecular embeddings shape: torch.Size([111, 384])


In [11]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_name)
print("Hidden size:", config.hidden_size)

Hidden size: 384


In [12]:
print("Tokenized input shape:", inputs["input_ids"].shape)

Tokenized input shape: torch.Size([760, 79])


In [13]:
pd2 = pd.DataFrame(embeddings_MTR)
pd2

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,0.098624,0.042223,-0.195243,0.209923,0.247522,-0.200116,0.130730,-0.078926,0.217784,-0.182055,...,-0.083785,-0.015257,0.208670,-0.099257,0.036652,0.011307,-0.015858,-0.034473,0.080340,0.090633
1,-0.195444,-0.006335,-0.125551,0.019175,-0.001245,-0.109488,0.118288,-0.345516,0.020197,-0.318505,...,0.153339,-0.094293,0.072913,-0.126550,0.012802,-0.071275,0.245102,-0.169543,-0.080592,-0.355988
2,0.019954,0.011441,-0.158951,0.221017,0.246666,-0.214977,0.122784,-0.182540,0.290653,-0.235725,...,-0.117349,-0.072634,0.212387,0.004530,-0.039211,-0.015519,0.015888,-0.153794,0.055760,0.118594
3,0.208809,0.168003,-0.015691,-0.009882,0.152641,0.031507,-0.038857,-0.054616,0.173442,-0.060251,...,-0.128801,-0.175998,0.109642,-0.068736,0.190212,-0.180588,0.214342,-0.124051,-0.300908,-0.076737
4,-0.041897,-0.128232,-0.096629,-0.068780,0.078374,-0.041795,0.128524,0.017905,0.171965,-0.143606,...,0.128640,-0.195388,0.355525,-0.314848,0.265437,-0.197671,0.041610,0.207000,-0.186747,0.126672
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,-0.099143,0.104436,0.011019,-0.165326,0.135885,-0.002735,-0.053308,0.116025,0.044211,-0.000038,...,-0.215827,0.197586,0.226231,-0.037536,0.056611,-0.155834,-0.153474,0.095045,-0.010944,-0.130073
756,0.011156,0.046114,0.019320,0.022109,0.234958,0.037306,0.037335,0.020181,0.005356,-0.084363,...,-0.043167,0.147757,0.299443,0.175839,-0.089705,-0.102291,-0.110185,0.237602,0.147709,-0.066069
757,-0.224795,0.074084,-0.111290,-0.070322,0.037394,-0.100532,0.073335,-0.244298,0.019439,-0.383307,...,0.119012,-0.095294,0.093073,-0.222050,0.105182,-0.022975,0.171083,-0.198204,-0.140231,-0.261504
758,-0.104002,0.089321,-0.083992,0.218112,0.071453,-0.035407,0.231394,-0.124874,-0.125678,0.000591,...,0.142494,0.000984,0.231777,0.024092,0.036001,0.095570,-0.203001,-0.023051,0.020927,-0.080840


In [16]:
pd2.to_csv('chemberta_MTR.csv', index=False)

In [17]:
pd.read_csv('chemberta_MTR.csv')

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,0.098624,0.042223,-0.195243,0.209923,0.247522,-0.200116,0.130730,-0.078926,0.217784,-0.182055,...,-0.083785,-0.015257,0.208670,-0.099257,0.036652,0.011307,-0.015858,-0.034473,0.080340,0.090633
1,-0.195444,-0.006335,-0.125551,0.019175,-0.001245,-0.109488,0.118288,-0.345516,0.020197,-0.318505,...,0.153339,-0.094293,0.072913,-0.126550,0.012802,-0.071275,0.245102,-0.169543,-0.080592,-0.355988
2,0.019954,0.011441,-0.158951,0.221017,0.246666,-0.214977,0.122784,-0.182540,0.290653,-0.235725,...,-0.117349,-0.072634,0.212387,0.004530,-0.039211,-0.015519,0.015888,-0.153794,0.055760,0.118594
3,0.208809,0.168003,-0.015691,-0.009882,0.152641,0.031507,-0.038857,-0.054616,0.173442,-0.060251,...,-0.128801,-0.175998,0.109642,-0.068736,0.190212,-0.180588,0.214342,-0.124051,-0.300908,-0.076737
4,-0.041897,-0.128232,-0.096629,-0.068780,0.078374,-0.041795,0.128524,0.017905,0.171965,-0.143606,...,0.128640,-0.195388,0.355525,-0.314848,0.265437,-0.197671,0.041610,0.207000,-0.186747,0.126672
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,-0.099143,0.104436,0.011019,-0.165326,0.135885,-0.002735,-0.053308,0.116025,0.044211,-0.000038,...,-0.215827,0.197586,0.226231,-0.037536,0.056611,-0.155834,-0.153475,0.095045,-0.010944,-0.130073
756,0.011156,0.046114,0.019320,0.022109,0.234958,0.037306,0.037335,0.020181,0.005356,-0.084363,...,-0.043167,0.147757,0.299443,0.175839,-0.089705,-0.102291,-0.110185,0.237602,0.147709,-0.066069
757,-0.224795,0.074084,-0.111290,-0.070322,0.037394,-0.100532,0.073335,-0.244298,0.019439,-0.383307,...,0.119012,-0.095294,0.093073,-0.222050,0.105182,-0.022975,0.171083,-0.198204,-0.140231,-0.261504
758,-0.104002,0.089321,-0.083992,0.218112,0.071453,-0.035407,0.231394,-0.124874,-0.125678,0.000591,...,0.142494,0.000984,0.231777,0.024092,0.036001,0.095570,-0.203001,-0.023051,0.020927,-0.080840
